In [10]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('db/title.basics.tsv', sep='\t', na_values='\\N')

C:\Users\muniz\AppData\Local\Temp\ipykernel_26136\2934157030.py:1: DtypeWarning: Columns (0: runtimeMinutes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('db/title.basics.tsv', sep='\t', na_values='\\N')


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12402756 entries, 0 to 12402755
Data columns (total 9 columns):
 #   Column          Dtype  
---  ------          -----  
 0   tconst          str    
 1   titleType       str    
 2   primaryTitle    str    
 3   originalTitle   str    
 4   isAdult         int64  
 5   startYear       float64
 6   endYear         float64
 7   runtimeMinutes  object 
 8   genres          str    
dtypes: float64(2), int64(1), object(1), str(5)
memory usage: 1.6+ GB


In [7]:
df = df[df['titleType'] == 'movie']
df.drop(columns=['titleType', 'isAdult', 'endYear', 'runtimeMinutes'], inplace=True)

In [8]:
df.info()

<class 'pandas.DataFrame'>
Index: 742013 entries, 8 to 12402706
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   tconst         742013 non-null  str    
 1   primaryTitle   742010 non-null  str    
 2   originalTitle  742010 non-null  str    
 3   startYear      630906 non-null  float64
 4   genres         664247 non-null  str    
dtypes: float64(1), str(4)
memory usage: 73.3 MB


In [8]:
# df.dropna(subset=['genres'], inplace=True)

In [12]:
from functools import reduce

df['genres'] = df['genres'].apply(lambda l: [] if l is np.nan else l.split(','))
genres = list(reduce(lambda acc, x: set(acc) | set(x), df['genres'], set()))
print(genres)

['Adventure', 'Mystery', 'Music', 'Film-Noir', 'Talk-Show', 'Family', 'Thriller', 'Western', 'Crime', 'Musical', 'Horror', 'Biography', 'News', 'Action', 'Comedy', 'Animation', 'Fantasy', 'Sci-Fi', 'Reality-TV', 'Documentary', 'Sport', 'Drama', 'Game-Show', 'History', 'War', 'Romance', 'Adult']


In [13]:
from genres_mask import GENRE_BY_NAME, Genre

def genre_list_to_bitmask(genre_list: list[str]) -> int:
    mask = Genre(0)
    for genre in genre_list:
        g = GENRE_BY_NAME[genre]
        mask |= g
    return mask


In [14]:
df['genres'] = df['genres'].apply(genre_list_to_bitmask)

In [15]:
df.to_parquet('db/movies.parquet')